# 002 LangGraph Quickstart

这是 LangGraph 学习线的第二份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/quickstart

学习目标：

1. 跑通一个最小 calculator agent graph
2. 理解 tools、model node、tool node、conditional edge 的关系
3. 理解 `Annotated[list[AnyMessage], operator.add]` 为什么能追加消息
4. 学会用 `should_continue` 控制 agent loop 是否继续
5. 对比 Quickstart 的真实模型版本和本课 fake model 版本

官方 Quickstart 使用真实模型。为了让本仓库教学稳定可运行，本课使用 fake model 复现同样的控制流，不消耗 API 额度。

## 1. Quickstart 要跑通什么

官方 Quickstart 的核心不是“计算器”本身，而是这个 agent loop：

```text
user message
  -> model node
  -> 如果模型要调用工具，进入 tool node
  -> tool node 把结果写回 messages
  -> 回到 model node
  -> 如果模型不再调用工具，结束
```

这正是很多 tool-calling agent 的基本结构。

In [46]:
import importlib.metadata
import operator
from typing import Annotated, Literal

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

print("langgraph", importlib.metadata.version("langgraph"))

langgraph 1.2.1


## 2. 定义 calculator tools

工具是确定性函数。

这里定义三个工具：

- `add`
- `multiply`
- `divide`

本课演示只会调用 `add`，但真实 agent 可以按模型输出选择不同工具。

In [47]:
@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b


@tool
def divide(a: int, b: int) -> float:
    """Divides a by b."""
    return a / b


tools = [add, multiply, divide]
tools_by_name = {item.name: item for item in tools}

for item in tools:
    print(item.name, "->", item.description)

add -> Adds a and b.
multiply -> Multiplies a and b.
divide -> Divides a by b.


## 3. 定义 graph state

Quickstart 的 state 重点是 messages。

这里的写法比较特别：

```python
messages: Annotated[list[AnyMessage], operator.add]
```

可以先理解成：

```text
当节点返回新的 messages 时，不是覆盖旧 messages，而是追加到旧 messages 后面。
```

Java 类比：

```java
state.messages.addAll(nodeResult.messages);
```

`llm_calls` 是我们额外加的计数器，用来观察模型节点跑了几次。

In [48]:
class CalculatorState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

## 4. 用 fake model 模拟 tool calling

真实 Quickstart 会调用模型，让模型决定是否调用工具。

这里用 fake model 固定返回两步：

1. 第一次调用：请求 `add(a=3, b=4)`
2. 第二次调用：根据工具结果给最终回答

这样我们能稳定观察 graph 控制流。

In [49]:
class FakeCalculatorModel:
    def __init__(self):
        self.calls = 0

    def invoke(self, messages: list[AnyMessage]) -> AIMessage:
        self.calls += 1

        if self.calls == 1:
            return AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "add",
                        "args": {"a": 3, "b": 4},
                        "id": "call_1",
                    }
                ],
            )

        return AIMessage(content="3 + 4 = 7")

## 5. 定义 model node 和 tool node

LangGraph 里的 node 是普通函数。

这个例子有两个节点：

```text
llm_call  -> 调模型，可能产生 tool call
tool_node -> 执行模型请求的工具，把结果写回 messages
```

In [50]:
def build_calculator_agent():
    model = FakeCalculatorModel()

    def llm_call(state: CalculatorState) -> dict:
        response = model.invoke(state["messages"])
        return {
            "messages": [response],
            "llm_calls": state.get("llm_calls", 0) + 1,
        }

    def tool_node(state: CalculatorState) -> dict:
        result = []
        last_message = state["messages"][-1]

        for tool_call in last_message.tool_calls:
            selected_tool = tools_by_name[tool_call["name"]]
            observation = selected_tool.invoke(tool_call["args"])
            result.append(
                ToolMessage(
                    content=str(observation),
                    tool_call_id=tool_call["id"],
                )
            )

        return {"messages": result}

    def should_continue(state: CalculatorState) -> Literal["tool_node", "__end__"]:
        last_message = state["messages"][-1]
        if getattr(last_message, "tool_calls", None):
            return "tool_node"
        return END

    builder = StateGraph(CalculatorState)
    builder.add_node("llm_call", llm_call)
    builder.add_node("tool_node", tool_node)

    builder.add_edge(START, "llm_call")
    builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
    builder.add_edge("tool_node", "llm_call")

    return builder.compile()

## 6. 执行 calculator agent

图结构是：

```text
START
  -> llm_call
    -> tool_node -> llm_call
    -> END
```

注意 `llm_call` 会执行两次：

1. 第一次产生 tool call
2. 第二次产生最终答案

In [54]:
calculator_agent = build_calculator_agent()

calculator_result = calculator_agent.invoke(
    {
        "messages": [HumanMessage(content="Add 3 and 4.")],
        "llm_calls": 0,
    }
)

for message in calculator_result["messages"]:
    print(type(message).__name__, "=>", getattr(message, "content", ""))
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)

print("llm_calls:", calculator_result["llm_calls"])

HumanMessage => Add 3 and 4.
AIMessage => 
tool_calls: [{'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'call_1', 'type': 'tool_call'}]
ToolMessage => 7
AIMessage => 3 + 4 = 7
llm_calls: 2


## 7. 观察 stream updates

LangGraph 的一个重要能力是 streaming。

这里用 `stream_mode="updates"` 观察每个节点的 state update。

In [55]:
stream_agent = build_calculator_agent()

for chunk in stream_agent.stream(
    {
        "messages": [HumanMessage(content="Add 3 and 4.")],
        "llm_calls": 0,
    },
    stream_mode="updates",
):
    print(chunk)

{'llm_call': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'call_1', 'type': 'tool_call'}], invalid_tool_calls=[])], 'llm_calls': 1}}
{'tool_node': {'messages': [ToolMessage(content='7', tool_call_id='call_1')]}}
{'llm_call': {'messages': [AIMessage(content='3 + 4 = 7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'llm_calls': 2}}


## 8. 关键代码走读

这句定义了消息追加规则：

```python
messages: Annotated[list[AnyMessage], operator.add]
```

这句定义了模型节点后的条件跳转：

```python
builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
```

`should_continue` 的逻辑是：

```text
如果最后一条 AIMessage 有 tool_calls，去 tool_node。
否则结束。
```

`tool_node` 执行完后固定回到 `llm_call`：

```python
builder.add_edge("tool_node", "llm_call")
```

这就形成了最小 agent loop。

## 9. 和真实模型版本的区别

本课 fake model 版本：

- 不需要 API key
- tool call 是固定的
- 适合理解控制流

官方真实模型版本：

- 由模型根据用户问题决定是否调用工具
- 需要真实模型配置
- 更接近生产 agent

但两者的 graph 结构是同一种心智模型：

```text
model node -> conditional edge -> tool node -> model node -> END
```

## 10. 本讲练习

请用自己的话回答：

1. 为什么 `tool_node` 执行后要回到 `llm_call`？
2. `should_continue` 为什么要看最后一条消息？
3. 如果模型连续调用多个工具，`tool_node` 应该怎么处理？
4. `operator.add` 在 messages state 里起什么作用？

参考方向：

- 工具结果本身不是最终回答，模型还要基于工具结果生成面向用户的答案
- 最后一条 AI 消息决定当前是否需要工具
- 多个 tool call 可以逐个执行并返回多个 ToolMessage
- `operator.add` 让新 messages 追加到旧 messages 后面

## 11. 本讲小结

这一讲的核心：

```text
LangGraph Quickstart = 用图结构显式表达 tool-calling agent loop。
```

你现在应该能看懂：

- tool 如何注册到 `tools_by_name`
- model node 如何写入 AIMessage
- tool node 如何写入 ToolMessage
- conditional edge 如何控制是否继续
- stream updates 如何观察每个节点的输出

下一讲可以继续学习 LangGraph State 和 Reducer。